# Resolution characterization on the audited benchmark

The resolution simulation runs on real data: how far apart two detectors must sit in skill before the released labels can tell them apart. The synthetic study sweeps a δ-grid of detectors against clean labels and perturbs them with fixed noise tuples. Here the labels are the released ones, the candidate space is each change's modified node set, and the perturbation draws its noise from the audited per-stratum posteriors rather than from a grid.

The released labels carry both restrictions the population is defined by, so no exclusion is applied. The perturbation covers the positive branch only, thinning marked nodes by α and adding unmarked ones by γ⁺. The rates β and γ⁻ live on the negative region, which is not released, and stay out of scope.

A perturbed draw can leave a change without any defective candidate. Metrics defined over a pooled contingency table absorb such a change as false positives and keep the full population in every draw. Metrics defined as a mean over changes carry no value there and are averaged over the remaining changes, whose number is reported.

In [2]:
import json
from math import erf

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from google.cloud import storage

from benchmark_notebook import theme  # noqa: F401  registers thesis template
from benchmark_notebook.io import CloudStorage
from benchmark.attribution.history import VersionControl
from benchmark.definitions.partitions import Shard
from benchmark.localization.projection import NodeProjection
from benchmark.audit.sampling import HybridAuditSampling

## Apparatus and inputs

The binormal detector, the reported node sets, and the metrics in their pooled and per-change forms, followed by the stored assets the run reads: the stationary cohort that fixes the partition space, one syntax projection per repository, and the released label asset.

In [3]:
def predictions(p, n, mu, rng):
    # Binormal detector of skill delta: clean nodes ~ N(0,1), positive nodes ~ N(delta,1).
    pos = np.isin(np.arange(n), np.asarray(p, int))
    base = rng.standard_normal(n)
    scores = np.tile(base, (mu.size, 1))
    scores[:, pos] += mu[:, None]
    return scores


def confusion(order, y):
    # Cumulative counts along the descending-score order, one column per top-k.
    yy = y[order].astype(np.int64)
    P, n = int(y.sum()), y.size
    tp = np.cumsum(yy, axis=1)
    k = np.arange(1, n + 1)[None, :]
    fp = k - tp
    fn = P - tp
    tn = (n - P) - fp
    return tp, fp, fn, tn, P


def curve(name, tp, fp, fn, tn, P):
    # Metric at every operating point k = 0..n, vectorized over the mu rows.
    if name == "f1":
        den = 2 * tp + fp + fn
        body = np.where(den > 0, 2 * tp / np.maximum(den, 1), 1.0)
        zero = 1.0 if P == 0 else 0.0
    else:
        num = tp * tn - fp * fn
        den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        body = np.where(den > 0, num / np.maximum(den, 1e-12), 0.0)
        zero = 0.0
    return np.concatenate([np.full((tp.shape[0], 1), zero), body], axis=1)


def selection(name, S, y, order):
    # Node set the detector reports, fixed at the threshold that maximises the metric
    # on the released labels. The set does not change when the labels are perturbed.
    tp, fp, fn, tn, P = confusion(order, y)
    kstar = curve(name, tp, fp, fn, tn, P).argmax(axis=1)
    pred = np.zeros(S.shape, bool)
    for r, k in enumerate(kstar):
        if k > 0:
            pred[r, order[r, :k]] = True
    return pred


def pooled_f1(pred, ym):
    TP = (pred & ym).sum(axis=1)
    FP = (pred & ~ym).sum(axis=1)
    FN = (~pred & ym).sum(axis=1)
    return 2 * TP / np.maximum(2 * TP + FP + FN, 1)


def pooled_mcc(pred, ym, total):
    TP = (pred & ym).sum(axis=1).astype(float)
    FP = (pred & ~ym).sum(axis=1).astype(float)
    FN = (~pred & ym).sum(axis=1).astype(float)
    TN = total - TP - FP - FN
    num = TP * TN - FP * FN
    den = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    return np.where(den > 0, num / np.maximum(den, 1e-12), 0.0)


def pooled_auroc(ranks, ym, total):
    # Mann-Whitney AUC over one ranking of every node in the corpus.
    n_pos = int(ym.sum())
    n_neg = total - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.full(ranks.shape[0], np.nan)
    rank_sum = ranks[:, ym].sum(axis=1)
    return (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def change_ap(hit, n_pos):
    # Average precision of one change, averaged over the ranks carrying a positive.
    cum = np.cumsum(hit, axis=1)
    pos = np.arange(1, hit.shape[1] + 1)[None, :]
    return (cum / pos * hit).sum(axis=1) / n_pos


def change_rr(hit):
    # Reciprocal rank of the first positive of one change.
    return 1.0 / (np.argmax(hit, axis=1) + 1)


POOLED = ("f1", "mcc", "auroc")
PER_CHANGE = ("map", "mrr")


def theta_star(C, mu, counts, gamma, M=300, rng=None):
    """Resolution of every metric in one pass over the draws.

    Pooled metrics are computed once per draw over the whole corpus, so every change
    enters every draw and no case is dropped. Per-change metrics are averaged over
    changes, and a change whose perturbed label set is empty carries no value for them
    and is left out. The number of changes that remain is reported per draw.
    Candidate scores and the reported node sets stay fixed, only the labels change.
    """
    rng = rng or np.random.default_rng()
    sizes = [n for _, n, _ in C]
    offset = np.cumsum([0] + sizes)
    total = int(offset[-1])
    n_mu = mu.size

    pred = {name: np.zeros((n_mu, total), bool) for name in ("f1", "mcc")}
    ranks = np.empty((n_mu, total))
    order, Y = [], []
    for i, (p, n, _) in enumerate(C):
        S = predictions(p, n, mu, rng)
        y = np.isin(np.arange(n), list(p))
        o = np.argsort(-S, axis=1)
        a, b = offset[i], offset[i + 1]
        for name in ("f1", "mcc"):
            pred[name][:, a:b] = selection(name, S, y, o)
        ranks[:, a:b] = S
        order.append(o)
        Y.append(y)

    ascending = np.argsort(ranks, axis=1)
    rows = np.arange(n_mu)[:, None]
    ranks[rows, ascending] = np.arange(1, total + 1)[None, :]

    node_bin = [np.asarray(b, int) for _, _, b in C]
    strata = sorted(counts)
    alpha_ab = np.array([[1 + counts[j][0], 1 + counts[j][1]] for j in strata], float)
    gamma_ab = np.array([[1 + gamma[j][0], 1 + gamma[j][1]] for j in strata], float)
    index = {j: k for k, j in enumerate(strata)}

    agg = {name: np.empty((M, n_mu)) for name in POOLED + PER_CHANGE}
    evaluable = np.empty(M)

    for m in range(M):
        alpha_j = rng.beta(alpha_ab[:, 0], alpha_ab[:, 1])
        gamma_j = rng.beta(gamma_ab[:, 0], gamma_ab[:, 1])
        ym_all = np.empty(total, bool)
        ap = np.full((len(C), n_mu), np.nan)
        rr = np.full((len(C), n_mu), np.nan)
        kept = 0
        for i, y in enumerate(Y):
            bins = [index[j] for j in node_bin[i]]
            # alpha removes marked nodes, gamma+ adds unmarked ones.
            keep = rng.random(y.size) > alpha_j[bins]
            add = rng.random(y.size) < gamma_j[bins]
            ym = np.where(y, keep, add)
            ym_all[offset[i]:offset[i + 1]] = ym
            n_pos = int(ym.sum())
            if n_pos == 0:
                continue
            kept += 1
            hit = ym[order[i]]
            ap[i] = change_ap(hit, n_pos)
            rr[i] = change_rr(hit)

        evaluable[m] = kept
        agg["f1"][m] = pooled_f1(pred["f1"], ym_all)
        agg["mcc"][m] = pooled_mcc(pred["mcc"], ym_all, total)
        agg["auroc"][m] = pooled_auroc(ranks, ym_all, total)
        agg["map"][m] = np.nanmean(ap, axis=0)
        agg["mrr"][m] = np.nanmean(rr, axis=0)

    out = {}
    for name in POOLED + PER_CHANGE:
        lo, hi = np.nanquantile(agg[name], [0.025, 0.975], axis=0)
        out[name] = (np.nanmean(agg[name], axis=0), hi - lo, float(np.nanmax(hi - lo)))
    out["evaluable"] = (evaluable.mean(), evaluable.min(), evaluable.max(), len(C))
    return out

In [4]:
store = CloudStorage(bucket="bug-request-bucket")
client = storage.Client(project="bug-request")
stationary = pl.from_arrow(
    store.asset(
        key="scope-repository-stationary",
        location="main",
        version="1.0.0",
        partition="default",
        columns=["name"],
    )
)
names = sorted(stationary["name"].to_list())
sampler = HybridAuditSampling(fraction=0.75, bins=3, seed=451600)


class Repositories:
    # One NodeProjection per repository, built lazily and reused for both the
    # candidate node set N(c) and the failed-change marking lift.
    def __init__(self):
        self._projections = {}

    def projection(self, name):
        if name not in self._projections:
            self._projections[name] = NodeProjection(
                VersionControl.create(
                    client,
                    bucket="bug-request-bucket",
                    key="scope-source-raw",
                    location="main",
                    version="4.0.0",
                    partition=Shard.to_space(names, name),
                )
            )
        return self._projections[name]


repos = Repositories()
label = pl.from_arrow(
    store.asset(
        key="localization-change-label",
        version="1.0.0",
        location="main",
        partition="default",
    )
)
print(f"defects={label.height}")

defects=360


## Per-stratum $α$ counters

The counters the simulation perturbs with are derived during the audit but never written to an asset, so they are rebuilt from the same annotation and trace inputs. All three resolutions are built because the released labels can be consumed at any of them, and the audited error rate differs between them. The audited changes are cut to the released population first, so the counters and the candidate spaces below describe the same set of changes.

In [5]:
# Reproduce the unconditional alpha counters as the audit derives them: passed changes
# contribute their audited node decisions, failed changes contribute their pipeline
# markings (all incorrect, Y_c = empty). a_j = rejected + fail, b_j = accepted.
annotation = pl.from_arrow(
    store.asset(
        key="audit-change-annotation",
        version="1.0.0",
        location="main",
        partition="default",
    )
).with_columns(
    pl.when(~pl.col("categories").list.contains("intrinsic"))
    .then(pl.lit("stage_1"))
    .when(pl.col("attribution") == "wrong")
    .then(pl.lit("stage_2"))
    .otherwise(pl.lit("passed"))
    .alias("stage")
)

# The released population is the label asset: it applies the single-branch and
# language-scope restrictions itself, so the audited changes are cut to it by a
# semi-join rather than by repeating either rule here.
annotation = annotation.join(
    label.select(pl.col("defect").alias("introducer")),
    on="introducer",
    how="semi",
)
print(f"audited changes in the released population: {annotation.height}")

# Passed changes: lift each selection marking to its node.
mark_rows = []
for row in annotation.filter(pl.col("stage") == "passed").iter_rows(named=True):
    projection = repos.projection(row["repository"])
    for block in json.loads(row["selections"]):
        origin = "manual" if block["origin"] == "manual" else "suggested"
        status = "accepted" if origin == "manual" else block["status"]
        for line in block["lines"]:
            node = projection.project(
                row["introducer"], block["path"], line["line"], line["side"]
            )
            mark_rows.append(
                {
                    "repository": row["repository"],
                    "number": row["number"],
                    "rho_bin": row["rho_bin"],
                    "path": block["path"],
                    "origin": origin,
                    "status": status,
                    "start": None if node is None else list(node.start),
                    "end": None if node is None else list(node.end),
                    "type": None if node is None else node.type,
                }
            )
marks = pl.DataFrame(mark_rows)

# A marked node is accepted once any of its suggested lines was accepted, since one
# true defect line makes the node defective. Only when none was accepted does a
# rejection decide, and only when neither occurs does a manual addition remain.
node_marks = (
    marks.filter(pl.col("type").is_not_null())
    .group_by("repository", "number", "rho_bin", "path", "start", "end", "type")
    .agg(
        ((pl.col("status") == "accepted") & (pl.col("origin") == "suggested"))
        .any()
        .alias("has_accepted"),
        ((pl.col("status") == "rejected") & (pl.col("origin") == "suggested"))
        .any()
        .alias("has_rejected"),
        (pl.col("origin") == "manual").any().alias("has_added"),
    )
    .with_columns(
        pl.when(pl.col("has_accepted"))
        .then(pl.lit("accepted"))
        .when(pl.col("has_rejected"))
        .then(pl.lit("rejected"))
        .when(pl.col("has_added"))
        .then(pl.lit("added"))
        .alias("decision")
    )
)
assert node_marks["decision"].null_count() == 0, "a marked node carries no decision"

# Failed changes: pipeline markings from the trace, all incorrect.
sample = pl.from_arrow(
    store.asset(key="audit-change-sample", version="4.0.0", partition="default")
).select(
    pl.col("name").alias("repository"),
    "correction",
    pl.col("defect").alias("introducer"),
)
trace = pl.from_arrow(
    store.asset(
        key="localization-change-trace",
        version="3.1.0",
        partition="*",
        columns=["name", "correction", "defect", "locations"],
    )
).unique(subset=["name", "correction", "defect"])
failed = (
    annotation.filter(pl.col("stage") != "passed")
    .join(sample, on=["repository", "introducer"], how="left")
    .join(
        trace.select(
            pl.col("name").alias("repository"),
            "correction",
            pl.col("defect").alias("introducer"),
            "locations",
        ),
        on=["repository", "correction", "introducer"],
        how="left",
    )
)

fail_node_rows, fail_file_rows = [], []
for row in failed.iter_rows(named=True):
    projection = repos.projection(row["repository"])
    seen_nodes, seen_files = set(), set()
    for m in json.loads(row["locations"] or "[]"):
        token = str(m["line"])
        side = "base" if token[:1] == "D" else "after"
        number = token[1:] if token[:1] in ("L", "D") else token
        if not number.isdigit():
            continue
        node = projection.project(row["introducer"], m["file"], int(number), side)
        if node is None:
            continue
        key = (node.path, node.start, node.end, node.type)
        if key not in seen_nodes:
            seen_nodes.add(key)
            fail_node_rows.append({"rho_bin": row["rho_bin"]})
        if m["file"] not in seen_files:
            seen_files.add(m["file"])
            fail_file_rows.append(
                {
                    "repository": row["repository"],
                    "rho_bin": row["rho_bin"],
                    "number": row["number"],
                    "path": m["file"],
                }
            )

fail_nodes = (
    pl.DataFrame(fail_node_rows).group_by("rho_bin").len().rename({"len": "fail"})
)


def counters(passed_frame, fail_frame):
    return (
        passed_frame.join(fail_frame, on="rho_bin", how="left")
        .fill_null(0)
        .with_columns(
            (pl.col("rejected") + pl.col("fail")).alias("a_j"),
            pl.col("accepted").alias("b_j"),
        )
        .sort("rho_bin")
    )


# Node level.
passed_nodes = (
    node_marks.filter(pl.col("decision") != "added")
    .group_by("rho_bin")
    .agg(
        (pl.col("decision") == "rejected").sum().alias("rejected"),
        (pl.col("decision") == "accepted").sum().alias("accepted"),
    )
)
counts_node_frame = counters(passed_nodes, fail_nodes)

# File level under the same rule as the node level: a file is defective once any of
# its lines is, so one accepted line makes the file correctly marked. Manual additions
# form their own state and leave the alpha counters afterwards.
passed_modules = (
    marks.group_by("repository", "number", "rho_bin", "path")
    .agg(
        ((pl.col("status") == "accepted") & (pl.col("origin") == "suggested"))
        .any()
        .alias("has_accepted"),
        ((pl.col("status") == "rejected") & (pl.col("origin") == "suggested"))
        .any()
        .alias("has_rejected"),
        (pl.col("origin") == "manual").any().alias("has_added"),
    )
    .with_columns(
        pl.when(pl.col("has_accepted"))
        .then(pl.lit("accepted"))
        .when(pl.col("has_rejected"))
        .then(pl.lit("rejected"))
        .when(pl.col("has_added"))
        .then(pl.lit("added"))
        .alias("decision")
    )
    .filter(pl.col("decision") != "added")
    .group_by("rho_bin")
    .agg(
        (pl.col("decision") == "rejected").sum().alias("rejected"),
        (pl.col("decision") == "accepted").sum().alias("accepted"),
    )
)
fail_files = (
    pl.DataFrame(fail_file_rows)
    .unique(subset=["repository", "number", "path"])
    .group_by("rho_bin")
    .len()
    .rename({"len": "fail"})
)
counts_file_frame = counters(passed_modules, fail_files)

counts_node = {
    r["rho_bin"]: (r["a_j"], r["b_j"]) for r in counts_node_frame.iter_rows(named=True)
}
counts_file = {
    r["rho_bin"]: (r["a_j"], r["b_j"]) for r in counts_file_frame.iter_rows(named=True)
}


def report(title, counts, a_name, b_name):
    # a and b are the Beta shape counters. For alpha they count wrong against right
    # markings, for gamma+ omitted against rightly unmarked candidates. The mean is
    # the expectation of the Beta(1 + a, 1 + b) posterior the simulation draws from,
    # which departs from the raw ratio whenever the counters are small.
    for j, (a, b) in sorted(counts.items()):
        print(
            f"{title}  stratum {j}: a={a:6d} {a_name:19s} "
            f"b={b:6d} {b_name:19s} mean={(1 + a) / (2 + a + b):.3f}"
        )


report("alpha  node", counts_node, "rejected or failed", "accepted")
report("alpha  file", counts_file, "rejected or failed", "accepted")


# Omission γ⁺ counters, reproduced over the audited changes as the audit derives them:
# N(c) is enumerated per audited change from the introducer diff, a_j = added nodes,
# b_j = the rightly-unmarked nodes |N(c)| - |Ỹ_c| - |A_c|. Node and file level,
# unconditional (fails contribute their trace markings with A_c = 0).
#
# N(c) keys on the span (path, start, end) only, matching the label asset: a block and
# the statement filling it are one candidate. Ỹ_c below is subtracted from this N(c),
# so it keys the same way.
def change_nodes(repository, commit):
    projection = repos.projection(repository)
    vc = projection._repository
    base = vc.parents(commit)[0]
    entries = set()
    for patch in vc.diff(base, commit, context=0):
        new_path, old_path = patch.delta.new_file.path, patch.delta.old_file.path
        for hunk in patch.hunks:
            for entry in hunk.lines:
                if entry.origin == "-":
                    node = projection.project(
                        commit, old_path, entry.old_lineno, "base"
                    )
                elif entry.origin == "+":
                    node = projection.project(
                        commit, new_path, entry.new_lineno, "after"
                    )
                else:
                    continue
                if node is not None:
                    entries.add((node.path, node.start, node.end))
    return entries


audit_nc = {
    (row["repository"], row["number"]): change_nodes(
        row["repository"], row["introducer"]
    )
    for row in annotation.iter_rows(named=True)
}
audit_n = pl.DataFrame(
    [
        {
            "repository": repo,
            "number": number,
            "n_c": len(nodes),
            "n_files": len({k[0] for k in nodes}),
        }
        for (repo, number), nodes in audit_nc.items()
    ]
)

passed_nodes_gamma = (
    node_marks.group_by("repository", "number", "rho_bin")
    .agg(
        (pl.col("decision") != "added").sum().alias("marked"),
        (pl.col("decision") == "added").sum().alias("added"),
    )
    .join(
        audit_n.select("repository", "number", "n_c"),
        on=["repository", "number"],
        how="left",
    )
    .group_by("rho_bin")
    .agg(
        pl.col("added").sum().alias("a_j"),
        (pl.col("n_c") - pl.col("marked") - pl.col("added")).sum().alias("b_j"),
    )
)

# File level: added files = files carrying a manual node not already suggested.
suggested_files = {
    (r["repository"], r["number"], r["path"])
    for r in node_marks.filter(pl.col("decision") != "added").iter_rows(named=True)
}
added_files = {
    (r["repository"], r["number"], r["path"])
    for r in node_marks.filter(pl.col("decision") == "added").iter_rows(named=True)
} - suggested_files
added_files_frame = pl.DataFrame(
    [{"repository": repo, "number": num, "path": p} for (repo, num, p) in added_files],
    schema={"repository": pl.Utf8, "number": pl.Int64, "path": pl.Utf8},
)
passed_files_gamma = (
    node_marks.select("repository", "number", "rho_bin")
    .unique()
    .join(
        added_files_frame.group_by("repository", "number")
        .len()
        .rename({"len": "added"}),
        on=["repository", "number"],
        how="left",
    )
    .join(
        node_marks.filter(pl.col("decision") != "added")
        .group_by("repository", "number")
        .agg(pl.col("path").n_unique().alias("marked_files")),
        on=["repository", "number"],
        how="left",
    )
    .join(
        audit_n.select("repository", "number", "n_files"),
        on=["repository", "number"],
        how="left",
    )
    .fill_null(0)
    .group_by("rho_bin")
    .agg(
        pl.col("added").sum().alias("a_j"),
        (pl.col("n_files") - pl.col("marked_files") - pl.col("added"))
        .sum()
        .alias("b_j"),
    )
)

# Fails contribute rightly-unmarked (A_c = 0): b_j += n_c - |Ỹ_c_trace|. Ỹ_c keys on
# the span only, so it aligns with the span-only N(c) it is subtracted from.
fail_gamma_rows = []
for row in failed.iter_rows(named=True):
    projection = repos.projection(row["repository"])
    nodes, files_seen = set(), set()
    for m in json.loads(row["locations"] or "[]"):
        token = str(m["line"])
        side = "base" if token[:1] == "D" else "after"
        num = token[1:] if token[:1] in ("L", "D") else token
        if not num.isdigit():
            continue
        node = projection.project(row["introducer"], m["file"], int(num), side)
        if node is None:
            continue
        nodes.add((node.path, node.start, node.end))
        files_seen.add(m["file"])
    universe = audit_nc.get((row["repository"], row["number"]), set())
    fail_gamma_rows.append(
        {
            "rho_bin": row["rho_bin"],
            "n_c": len(universe),
            "y_nodes": len(nodes),
            "n_files": len({k[0] for k in universe}),
            "y_files": len(files_seen),
        }
    )
fail_gamma = pl.DataFrame(fail_gamma_rows)
fail_node_gamma = fail_gamma.group_by("rho_bin").agg(
    (pl.col("n_c") - pl.col("y_nodes")).sum().alias("b_j")
)
fail_file_gamma = fail_gamma.group_by("rho_bin").agg(
    (pl.col("n_files") - pl.col("y_files")).sum().alias("b_j")
)


def merge_gamma(passed, fail):
    return (
        passed.join(fail, on="rho_bin", how="left", suffix="_fail")
        .fill_null(0)
        .with_columns((pl.col("b_j") + pl.col("b_j_fail")).alias("b_j"))
        .select("rho_bin", "a_j", "b_j")
        .sort("rho_bin")
    )


gamma_node = {
    r["rho_bin"]: (r["a_j"], r["b_j"])
    for r in merge_gamma(passed_nodes_gamma, fail_node_gamma).iter_rows(named=True)
}
gamma_file = {
    r["rho_bin"]: (r["a_j"], r["b_j"])
    for r in merge_gamma(passed_files_gamma, fail_file_gamma).iter_rows(named=True)
}
report("gamma+ node", gamma_node, "added", "rightly unmarked")
report("gamma+ file", gamma_file, "added", "rightly unmarked")

audited changes in the released population: 98
alpha  node  stratum 0: a=    11 rejected or failed  b=    14 accepted            mean=0.444
alpha  node  stratum 1: a=    54 rejected or failed  b=    65 accepted            mean=0.455
alpha  node  stratum 2: a=    41 rejected or failed  b=    39 accepted            mean=0.512
alpha  file  stratum 0: a=     6 rejected or failed  b=    12 accepted            mean=0.350
alpha  file  stratum 1: a=    16 rejected or failed  b=    39 accepted            mean=0.298
alpha  file  stratum 2: a=    12 rejected or failed  b=    32 accepted            mean=0.283
gamma+ node  stratum 0: a=     1 added               b=  3684 rightly unmarked    mean=0.001
gamma+ node  stratum 1: a=     2 added               b= 12658 rightly unmarked    mean=0.000
gamma+ node  stratum 2: a=     2 added               b=  7155 rightly unmarked    mean=0.000
gamma+ file  stratum 0: a=     1 added               b=   105 rightly unmarked    mean=0.019
gamma+ file  stratum 1:

In [6]:
# Line-level counters. The audit records one decision per marked line before those
# decisions are lifted to nodes, so the alpha counters at line resolution come from the
# same frame without the node aggregation. The candidate space is every added or deleted
# line of the defect diff that maps to a syntax node, the same filter the node level
# applies.

# alpha, passed changes. Manual markings are additions and belong to the omission
# counters, so only suggested lines enter here.
passed_lines = (
    marks.filter(pl.col("origin") == "suggested")
    .group_by("rho_bin")
    .agg(
        (pl.col("status") == "rejected").sum().alias("rejected"),
        (pl.col("status") == "accepted").sum().alias("accepted"),
    )
)

# alpha, failed changes. Every distinct marked line of the trace counts as wrong.
fail_line_rows = []
for row in failed.iter_rows(named=True):
    seen = set()
    for m in json.loads(row["locations"] or "[]"):
        token = (m["file"], str(m["line"]))
        if token not in seen:
            seen.add(token)
            fail_line_rows.append({"rho_bin": row["rho_bin"]})
fail_lines = (
    pl.DataFrame(fail_line_rows).group_by("rho_bin").len().rename({"len": "fail"})
)
counts_line_frame = counters(passed_lines, fail_lines)
counts_line = {
    r["rho_bin"]: (r["a_j"], r["b_j"]) for r in counts_line_frame.iter_rows(named=True)
}


def change_line_count(repository, commit):
    """Edited lines of a change that map to a syntax node."""
    projection = repos.projection(repository)
    vc = projection._repository
    base = vc.parents(commit)[0]
    seen = set()
    for patch in vc.diff(base, commit, context=0):
        new_path, old_path = patch.delta.new_file.path, patch.delta.old_file.path
        for hunk in patch.hunks:
            for entry in hunk.lines:
                if entry.origin == "-":
                    key = (old_path, "base", entry.old_lineno)
                    node = projection.project(commit, old_path, entry.old_lineno, "base")
                elif entry.origin == "+":
                    key = (new_path, "after", entry.new_lineno)
                    node = projection.project(commit, new_path, entry.new_lineno, "after")
                else:
                    continue
                if node is not None:
                    seen.add(key)
    return len(seen)


audit_lines = {
    (row["repository"], row["number"]): change_line_count(
        row["repository"], row["introducer"]
    )
    for row in annotation.iter_rows(named=True)
}
audit_line_n = pl.DataFrame(
    [
        {"repository": repo, "number": number, "n_lines": count}
        for (repo, number), count in audit_lines.items()
    ]
)

# gamma+ at line resolution, added lines against the lines rightly left unmarked.
passed_lines_gamma = (
    marks.group_by("repository", "number", "rho_bin")
    .agg(
        (pl.col("origin") == "suggested").sum().alias("marked"),
        (pl.col("origin") == "manual").sum().alias("added"),
    )
    .join(audit_line_n, on=["repository", "number"], how="left")
    .fill_null(0)
    .group_by("rho_bin")
    .agg(
        pl.col("added").sum().alias("a_j"),
        (pl.col("n_lines") - pl.col("marked") - pl.col("added")).sum().alias("b_j"),
    )
)

fail_line_gamma_rows = []
for row in failed.iter_rows(named=True):
    seen = set()
    for m in json.loads(row["locations"] or "[]"):
        seen.add((m["file"], str(m["line"])))
    fail_line_gamma_rows.append(
        {
            "rho_bin": row["rho_bin"],
            "n_lines": audit_lines.get((row["repository"], row["number"]), 0),
            "y_lines": len(seen),
        }
    )
fail_line_gamma = (
    pl.DataFrame(fail_line_gamma_rows)
    .group_by("rho_bin")
    .agg((pl.col("n_lines") - pl.col("y_lines")).sum().alias("b_j"))
)

gamma_line = {
    r["rho_bin"]: (r["a_j"], r["b_j"])
    for r in merge_gamma(passed_lines_gamma, fail_line_gamma).iter_rows(named=True)
}

report("alpha  line", counts_line, "rejected or failed", "accepted")
report("gamma+ line", gamma_line, "added", "rightly unmarked")

alpha  line  stratum 0: a=    14 rejected or failed  b=    23 accepted            mean=0.385
alpha  line  stratum 1: a=    86 rejected or failed  b=   106 accepted            mean=0.448
alpha  line  stratum 2: a=    56 rejected or failed  b=    62 accepted            mean=0.475
gamma+ line  stratum 0: a=     3 added               b= 10831 rightly unmarked    mean=0.000
gamma+ line  stratum 1: a=     5 added               b= 24951 rightly unmarked    mean=0.000
gamma+ line  stratum 2: a=     3 added               b= 23454 rightly unmarked    mean=0.000


## $Ỹ_c$ at node, file and line granularity

Each released change contributes a candidate space and the labelled subset within it. At node resolution the space is every modified syntax node of the change's diff, at file resolution every modified file, at line resolution every added or deleted line that maps to a syntax node. A file counts as labelled once it carries a labelled node, a line once a correction marked it. Every candidate carries the stratum of the correction that localised it, so the perturbation draws the stratum-specific rate. An unlabelled candidate takes the change's median labelled ρ, since it has no correction of its own.

In [7]:
# For each defect two candidate spaces are built from the same diff. At node level,
# N(c) is every modified syntax node (deduplicated), Ỹ_c the labelled nodes. At file
# level, N(c) is every modified .py file, Ỹ_c the files carrying a labelled node. Each
# candidate carries the rho_bin of the correction that localised it. Unlabelled ones
# take the defect's median labelled rho.
#
# A node's identity is its span (path, start, end), not its type: a block and the
# single statement that fills it share the same bytes and are one indistinguishable
# candidate. This matches the label asset, which merges such spans to one node.
label = label.with_columns(
    pl.col("label")
    .list.eval(pl.element().struct.field("name"))
    .list.first()
    .alias("name")
)

C_node, C_file = [], []
for row in label.iter_rows(named=True):
    projection = repos.projection(row["name"])
    vc = projection._repository  # noqa: SLF001 - reuse the warm repository handle
    defect = row["defect"]

    parents = vc.parents(defect)
    if not parents:
        continue
    base = parents[0]

    labelled = {
        (n["path"], tuple(n["start"]), tuple(n["end"])): n["rho"]
        for n in row["label"]
    }
    labelled_files = {}  # path to the list of rho values of its labelled nodes
    for (path, _, _), rho in labelled.items():
        labelled_files.setdefault(path, []).append(rho)

    universe = {}  # node key to rho
    files = {}  # path to the list of rho values of its modified nodes
    for patch in vc.diff(base, defect, context=0):
        new_path, old_path = patch.delta.new_file.path, patch.delta.old_file.path
        for hunk in patch.hunks:
            for entry in hunk.lines:
                if entry.origin == "+":
                    node = projection.project(
                        defect, new_path, entry.new_lineno, "after"
                    )
                elif entry.origin == "-":
                    node = projection.project(
                        defect, old_path, entry.old_lineno, "base"
                    )
                else:
                    continue
                if node is None:
                    continue
                key = (node.path, node.start, node.end)
                universe.setdefault(key, labelled.get(key))
                files.setdefault(node.path, [])

    for key, rho in labelled.items():  # released nodes are part of N(c)
        universe[key] = rho
    for path in labelled_files:
        files.setdefault(path, [])
    if not universe:
        continue

    fallback = float(np.median(list(labelled.values()))) if labelled else 0.0

    # node level
    keys = list(universe)
    node_pos = [i for i, k in enumerate(keys) if k in labelled]
    node_rho = np.array(
        [universe[k] if universe[k] is not None else fallback for k in keys]
    )
    C_node.append((np.array(node_pos), len(keys), sampler.strata(node_rho)))

    # file level: a file is positive if it carries a labelled node. Its rho is the
    # median rho of its labelled nodes, else the defect fallback.
    paths = list(files)
    file_pos = [i for i, p in enumerate(paths) if p in labelled_files]
    file_rho = np.array(
        [
            float(np.median(labelled_files[p])) if p in labelled_files else fallback
            for p in paths
        ]
    )
    C_file.append((np.array(file_pos), len(paths), sampler.strata(file_rho)))

node_sizes = np.array([n for _, n, _ in C_node])
file_sizes = np.array([n for _, n, _ in C_file])
print(f"changes={len(C_node)}")
print(
    f"  node  |N(c)| median={int(np.median(node_sizes))} max={node_sizes.max()}  positives median={int(np.median([len(p) for p,_,_ in C_node]))}"
)
print(
    f"  file  |N(c)| median={int(np.median(file_sizes))} max={file_sizes.max()}  positives median={int(np.median([len(p) for p,_,_ in C_file]))}"
)

changes=360
  node  |N(c)| median=100 max=10280  positives median=2
  file  |N(c)| median=6 max=191  positives median=1


In [8]:
# Candidate space at line resolution, built from the same diff walk. N(c) is every
# added or deleted line that maps to a syntax node, Y_c the lines a correction marked.
# A marked line carries the rho of its labelled node, an unmarked one the change median.
C_line = []
for row in label.iter_rows(named=True):
    projection = repos.projection(row["name"])
    vc = projection._repository  # noqa: SLF001 - reuse the warm repository handle
    defect = row["defect"]
    parents = vc.parents(defect)
    if not parents:
        continue
    base = parents[0]

    marked_rho = {}
    for n in row["label"]:
        for token in n["lines"]:
            side = "base" if token[:1] == "R" else "after"
            marked_rho[(n["path"], side, int(token[1:]))] = n["rho"]

    keys, seen = [], set()
    for patch in vc.diff(base, defect, context=0):
        new_path, old_path = patch.delta.new_file.path, patch.delta.old_file.path
        for hunk in patch.hunks:
            for entry in hunk.lines:
                if entry.origin == "+":
                    key = (new_path, "after", entry.new_lineno)
                    node = projection.project(defect, new_path, entry.new_lineno, "after")
                elif entry.origin == "-":
                    key = (old_path, "base", entry.old_lineno)
                    node = projection.project(defect, old_path, entry.old_lineno, "base")
                else:
                    continue
                if node is not None and key not in seen:
                    seen.add(key)
                    keys.append(key)

    for key in marked_rho:  # released lines are part of N(c)
        if key not in seen:
            seen.add(key)
            keys.append(key)
    if not keys:
        continue

    fallback = float(np.median(list(marked_rho.values()))) if marked_rho else 0.0
    line_pos = [i for i, k in enumerate(keys) if k in marked_rho]
    line_rho = np.array([marked_rho.get(k, fallback) for k in keys])
    C_line.append((np.array(line_pos), len(keys), sampler.strata(line_rho)))

line_sizes = np.array([n for _, n, _ in C_line])
print(f"changes={len(C_line)}")
print(
    f"  line  |N(c)| median={int(np.median(line_sizes))} max={line_sizes.max()}  "
    f"positives median={int(np.median([len(p) for p, _, _ in C_line]))}"
)

changes=360
  line  |N(c)| median=314 max=15301  positives median=3


## Edited lines to node classes: where defect lines land

Every added or deleted line of a defect diff, context lines excluded, lifts to exactly
one node. A line is *defect* when a correction marked it, recorded as a token prefixed
L or R in the label struct, otherwise *clean*. Classifying each node by the lines that
lift to it splits N(c) into three: **pure-defect** (only defect lines), **mixed**
(defect and clean), **pure-clean** (only clean lines).

The question is contamination: how many clean lines a defect node drags in with it. A
mixed node is scored positive as a whole, yet most of its edited lines are clean. The
Sankey below traces the two line sources into the three node classes. Its middle band,
the clean lines flowing into mixed nodes, is the contamination the node granularity
pays.

Two invariants are asserted while building it: every edited line lifts to at most one
node, and every marked line is a real added or deleted line. If either breaks the
reading is unsound, so the cell raises rather than reporting silently.

In [9]:
# Base is every added or deleted line of the defect diff, context excluded. A line is
# defect when a correction marked it, recorded as a token prefixed L or R in the label
# struct, else clean. Each line lifts to exactly one node, which is asserted, and a
# node is classed by the lines it carries.
node_class = {"pure_defect": 0, "mixed": 0, "pure_clean": 0}
line_flow = {"pure_defect": [0, 0], "mixed": [0, 0], "pure_clean": [0, 0]}  # [defect, clean]
edit_lines = 0
defect_lines = 0
lines_without_node = 0

for row in label.iter_rows(named=True):
    projection = repos.projection(row["name"])
    vc = projection._repository  # noqa: SLF001 - warm handle
    defect = row["defect"]
    parents = vc.parents(defect)
    if not parents:
        continue
    base = parents[0]

    # marked (defect) lines from the label struct tokens: (path, side, line)
    marked = set()
    for n in row["label"]:
        for token in n["lines"]:
            side = "base" if token[:1] == "R" else "after"
            marked.add((n["path"], side, int(token[1:])))

    # every added or deleted line maps to exactly one node span, or None for blank,
    # comment and non-Python lines
    line_node = {}
    for patch in vc.diff(base, defect, context=0):
        new_path, old_path = patch.delta.new_file.path, patch.delta.old_file.path
        for hunk in patch.hunks:
            for entry in hunk.lines:
                if entry.origin == "+":
                    key = (new_path, "after", entry.new_lineno)
                    node = projection.project(defect, new_path, entry.new_lineno, "after")
                elif entry.origin == "-":
                    key = (old_path, "base", entry.old_lineno)
                    node = projection.project(defect, old_path, entry.old_lineno, "base")
                else:
                    continue
                span = None if node is None else (node.path, node.start, node.end)
                assert line_node.get(key, span) == span, f"line {key} lifts to two nodes"
                line_node[key] = span

    assert marked <= set(line_node), "a marked line is not an added or deleted line"

    # per node: how many defect against clean lines lift to it
    per_node = {}  # span to the pair [defect, clean]
    for key, span in line_node.items():
        edit_lines += 1
        is_defect = key in marked
        defect_lines += is_defect
        if span is None:
            lines_without_node += 1
            continue
        agg = per_node.setdefault(span, [0, 0])
        agg[0 if is_defect else 1] += 1

    for span, (d, c) in per_node.items():
        cls = "mixed" if d and c else "pure_defect" if d else "pure_clean"
        node_class[cls] += 1
        line_flow[cls][0] += d
        line_flow[cls][1] += c

total_nodes = sum(node_class.values())
print(f"edited lines: {edit_lines}  defect {defect_lines} ({defect_lines / edit_lines:.1%})  "
      f"clean {edit_lines - defect_lines}  without node {lines_without_node}")
print(f"nodes: {total_nodes}")
for cls in ("pure_defect", "mixed", "pure_clean"):
    d, c = line_flow[cls]
    print(f"  {cls:11s} {node_class[cls]:6d} nodes ({node_class[cls] / total_nodes:5.1%})  "
          f"swallows {d:5d} defect + {c:6d} clean lines")

edited lines: 357739  defect 2265 (0.6%)  clean 355474  without node 103614
nodes: 120451
  pure_defect    911 nodes ( 0.8%)  swallows   982 defect +      0 clean lines
  mixed          653 nodes ( 0.5%)  swallows  1283 defect +  16964 clean lines
  pure_clean  118887 nodes (98.7%)  swallows     0 defect + 234896 clean lines


In [10]:
# Sankey: the two line sources, defect and clean, flow into the three node classes. The
# band from clean lines into mixed nodes is the contamination, meaning clean lines
# scored positive because they share a node with a defect line.
colorway = pio.templates["thesis"].layout.colorway
defect_color, clean_color, mixed_color = colorway[3], colorway[0], colorway[1]

# nodes: 0 defect lines, 1 clean lines, then 2 pure-defect, 3 mixed, 4 pure-clean
labels = [
    f"defect lines ({defect_lines})",
    f"clean lines ({edit_lines - defect_lines - lines_without_node})",
    f"pure-defect nodes ({node_class['pure_defect']})",
    f"mixed nodes ({node_class['mixed']})",
    f"pure-clean nodes ({node_class['pure_clean']})",
]
node_x = [0.001, 0.001, 0.9, 0.9, 0.9]
node_y = [0.15, 0.7, 0.05, 0.35, 0.85]

# flows: (source, target, value, from_defect)
flows = [
    (0, 2, line_flow["pure_defect"][0], True),   # defect into pure-defect
    (0, 3, line_flow["mixed"][0], True),          # defect into mixed
    (1, 3, line_flow["mixed"][1], False),         # clean into mixed (contamination)
    (1, 4, line_flow["pure_clean"][1], False),    # clean into pure-clean
]


def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        label=labels,
        x=node_x,
        y=node_y,
        color=[defect_color, clean_color, defect_color, mixed_color, clean_color],
        pad=20,
        thickness=18,
    ),
    link=dict(
        source=[f[0] for f in flows],
        target=[f[1] for f in flows],
        value=[f[2] for f in flows],
        color=[rgba(defect_color if f[3] else clean_color, 0.45) for f in flows],
    ),
))
fig.update_layout(template="thesis", width=760, height=420,
                  title="edited lines to node classes").show(
    config={"toImageButtonOptions": {"format": "svg"}})

into_mixed = line_flow["mixed"][0]
contamination = line_flow["mixed"][1]
print(f"defect lines into mixed nodes:  {into_mixed} of {defect_lines} "
      f"({into_mixed / defect_lines:.0%} of all defect lines)")
print(f"clean lines dragged in with them: {contamination}, the contamination the "
      f"node granularity pays")

defect lines into mixed nodes:  1283 of 2265 (57% of all defect lines)
clean lines dragged in with them: 16964, the contamination the node granularity pays


## Resolution $κ(δ)$

Resolution is the width of the interval the metric can occupy at a fixed detector skill once the label noise is drawn: at each δ the noise is resampled, the metric recomputed, and κ is the span between its 2.5th and 97.5th percentiles. Two detectors whose true skill differs by less than κ cannot be separated by the benchmark. One pass per resolution computes every metric on the same draws. The left panel gives the clean metric value, the right one the resolution. AUROC is reported rather than drawn, since under the binormal detector its clean value is the horizontal axis by construction.

In [11]:
grid = np.linspace(0.0, 14.0, 60)
auc = 0.5 * (1.0 + np.array([erf(d / 2.0) for d in grid]))
M = 1000
metrics = ["f1", "mcc", "auroc", "map", "mrr"]

# One pass per resolution computes every metric on the same draws, so the metrics are
# directly comparable. Pooled metrics keep all changes in every draw, per-change metrics
# report how many remained.
result = {}
for source, changes, counts, gamma in [
    ("node", C_node, counts_node, gamma_node),
    ("file", C_file, counts_file, gamma_file),
    ("line", C_line, counts_line, gamma_line),
]:
    result[source] = theta_star(
        changes, grid, counts, gamma, M=M, rng=np.random.default_rng(1)
    )
    mean, low, high, total = result[source]["evaluable"]
    print(
        f"{source}: each draw perturbs all {total} changes, on average {mean:.0f} "
        f"retain a defect and enter the per-change metrics (range {low:.0f} to {high:.0f})"
    )

node: each draw perturbs all 360 changes, on average 264 retain a defect and enter the per-change metrics (range 227 to 301)
file: each draw perturbs all 360 changes, on average 283 retain a defect and enter the per-change metrics (range 219 to 325)
line: each draw perturbs all 360 changes, on average 299 retain a defect and enter the per-change metrics (range 272 to 327)


In [12]:
colour = pio.templates["thesis"].layout.colorway[3]
shown = [
    ("f1", "F1", "solid"),
    ("mcc", "MCC", "dash"),
    ("map", "MAP", "dot"),
    ("mrr", "MRR", "dashdot"),
]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11)
for key, name, dash in shown:
    level, kappa, _ = result["node"][key]
    fig.add_trace(
        go.Scatter(
            x=auc, y=level, name=name, line=dict(color=colour, dash=dash, width=1.0)
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=auc,
            y=kappa,
            name=name,
            line=dict(color=colour, dash=dash, width=1.0),
            showlegend=False,
        ),
        row=1,
        col=2,
    )
fig.update_xaxes(title_text="Φ(δ/√2)", row=1, col=1).update_xaxes(
    title_text="Φ(δ/√2)", row=1, col=2
).update_yaxes(title_text="Metric", row=1, col=1).update_yaxes(
    title_text="κ(δ)", row=1, col=2
).update_layout(
    template="thesis", width=800, height=400,
).show(
    config={"toImageButtonOptions": {"format": "svg"}}
)

print("worst-case κ*:")
for name in metrics:
    values = "  ".join(f"{s}={result[s][name][2]:.3f}" for s in ("node", "file", "line"))
    print(f"  {name:6s} {values}")

print("\nAUROC over the grid, reported rather than drawn:")
for source in ("node", "file", "line"):
    kappa = result[source]["auroc"][1]
    print(f"  {source}: mean κ={kappa.mean():.4f}  sd={kappa.std():.4f}  max={kappa.max():.4f}")

worst-case κ*:
  f1     node=0.116  file=0.125  line=0.096
  mcc    node=0.099  file=0.125  line=0.082
  auroc  node=0.034  file=0.044  line=0.027
  map    node=0.093  file=0.062  line=0.088
  mrr    node=0.091  file=0.059  line=0.086

AUROC over the grid, reported rather than drawn:
  node: mean κ=0.0326  sd=0.0018  max=0.0335
  file: mean κ=0.0429  sd=0.0030  max=0.0443
  line: mean κ=0.0264  sd=0.0017  max=0.0273


## Checks on the sweep

Two properties the reading depends on. The first asks whether κ grows with skill rather than wandering with Monte Carlo jitter, reported as the largest single-step decrease against the overall rise. The second reads off the largest κ over a moderate skill band and the point where κ first exceeds a fixed level.

In [13]:
# Check: is kappa(delta) actually monotone, or just trending up with MC jitter?
# Reports the largest single-step decrease and the total "backwards" movement
# relative to the overall rise, per metric and resolution.
for source in ("node", "file", "line"):
    for name in metrics:
        kappa = result[source][name][1]
        steps = np.diff(kappa)
        drop = steps[steps < 0]
        max_drop = -drop.min() if drop.size else 0.0
        back = -drop.sum()
        rise = kappa[-1] - kappa[0]
        print(
            f"{source:4s} {name:6s}: max single drop={max_drop:.4f}  "
            f"total backwards={back:.4f}  net rise={rise:.4f}  "
            f"({(steps < 0).sum()} of {steps.size} steps negative)"
        )

node f1    : max single drop=0.0004  total backwards=0.0010  net rise=0.1095  (3 of 59 steps negative)
node mcc   : max single drop=0.0022  total backwards=0.0033  net rise=0.0897  (3 of 59 steps negative)
node auroc : max single drop=0.0006  total backwards=0.0006  net rise=0.0068  (17 of 59 steps negative)
node map   : max single drop=0.0018  total backwards=0.0028  net rise=0.0646  (4 of 59 steps negative)
node mrr   : max single drop=0.0018  total backwards=0.0070  net rise=0.0512  (7 of 59 steps negative)
file f1    : max single drop=0.0015  total backwards=0.0029  net rise=0.0606  (3 of 59 steps negative)
file mcc   : max single drop=0.0000  total backwards=0.0000  net rise=0.0814  (1 of 59 steps negative)
file auroc : max single drop=0.0007  total backwards=0.0011  net rise=0.0099  (8 of 59 steps negative)
file map   : max single drop=0.0013  total backwards=0.0024  net rise=0.0188  (5 of 59 steps negative)
file mrr   : max single drop=0.0022  total backwards=0.0066  net rise=0.

In [14]:
# Check: max kappa over the moderate skill range, node level, per metric.
# "Moderate" is read as AUC in [0.6, 0.9]. Adjust the band if a different range applies.
band = (auc >= 0.6) & (auc <= 0.9)
for name in metrics:
    kappa = result["node"][name][1]
    print(f"{name:6s}: max kappa in AUC [0.6, 0.9] = {kappa[band].max():.3f}")

f1    : max kappa in AUC [0.6, 0.9] = 0.034
mcc   : max kappa in AUC [0.6, 0.9] = 0.031
auroc : max kappa in AUC [0.6, 0.9] = 0.030
map   : max kappa in AUC [0.6, 0.9] = 0.063
mrr   : max kappa in AUC [0.6, 0.9] = 0.076


In [15]:
# Check: where kappa first exceeds 0.08, node level, per metric.
for name in metrics:
    kappa = result["node"][name][1]
    crossing = auc[kappa > 0.08]
    print(
        f"{name:6s}: kappa exceeds 0.08 from AUC = {crossing.min():.3f}"
        if crossing.size
        else f"{name:6s}: never exceeds 0.08"
    )

f1    : kappa exceeds 0.08 from AUC = 0.978
mcc   : kappa exceeds 0.08 from AUC = 0.991
auroc : never exceeds 0.08
map   : kappa exceeds 0.08 from AUC = 0.968
mrr   : kappa exceeds 0.08 from AUC = 0.934


In [16]:
# Measured ceiling. A detector that reproduces the pipeline markings exactly is scored
# against the audited truth. This needs no noise model, so it checks the value the
# simulation derives from the alpha posteriors against a direct observation.
for level, passed_frame, fail_frame in [
    ("node", passed_nodes, fail_nodes),
    ("file", passed_modules, fail_files),
    ("line", passed_lines, fail_lines),
]:
    merged = passed_frame.join(fail_frame, on="rho_bin", how="left").fill_null(0)
    TP = merged["accepted"].sum()
    FP = merged["rejected"].sum() + merged["fail"].sum()
    print(f"{level:5s} TP={TP:5d} FP={FP:5d}  F1 without omissions = {2*TP/(2*TP+FP):.3f}")

# Omissions are the manually added candidates, they enter as FN.
added_nodes = node_marks.filter(pl.col("decision") == "added").height
added_lines = marks.filter(pl.col("origin") == "manual").height
print(f"\nadded nodes {added_nodes}, added lines {added_lines}")

node  TP=  118 FP=  106  F1 without omissions = 0.690
file  TP=   83 FP=   34  F1 without omissions = 0.830
line  TP=  191 FP=  156  F1 without omissions = 0.710

added nodes 5, added lines 11


In [17]:
# Numbers the results chapter quotes but no cell above prints. Kept separate so the
# counter cells stay unchanged.
import statistics

# Markings of the passing changes, split into accepted and rejected, and the
# markings of the failed changes that enter the counters as wrong.
for level, passed_frame, fail_frame in [
    ("node", passed_nodes, fail_nodes),
    ("line", passed_lines, fail_lines),
    ("file", passed_modules, fail_files),
]:
    acc = int(passed_frame["accepted"].sum())
    rej = int(passed_frame["rejected"].sum())
    fail = int(fail_frame["fail"].sum())
    print(
        f"{level:5s} markings of passing changes: {acc + rej:5d}  "
        f"accepted {acc:5d}  rejected {rej:5d}   from failed changes {fail:5d}"
    )
print(f"nodes added by the annotator: {node_marks.filter(pl.col('decision') == 'added').height}")
print(f"lines added by the annotator: {marks.filter(pl.col('origin') == 'manual').height}")

# N(c) over the audited changes only. The cells above report it over all 360
# released changes, which is a different population.
audit_node_sizes = [len(v) for v in audit_nc.values()]
audit_line_sizes = list(audit_lines.values())
audit_file_sizes = audit_n["n_files"].to_list()
print(f"\nN(c) over the {len(audit_node_sizes)} audited changes")
for name, sizes in [
    ("nodes", audit_node_sizes),
    ("lines", audit_line_sizes),
    ("files", audit_file_sizes),
]:
    print(f"  {name}: median={int(statistics.median(sizes))} max={max(sizes)}")

# AUROC level across the grid, to see whether the metric traverses any range at all.
print("\nAUROC level across the grid")
for source in ("node", "file", "line"):
    level = result[source]["auroc"][0]
    print(f"  {source}: {level.min():.3f} to {level.max():.3f}")

node  markings of passing changes:   172  accepted   118  rejected    54   from failed changes    52
line  markings of passing changes:   284  accepted   191  rejected    93   from failed changes    63
file  markings of passing changes:    94  accepted    83  rejected    11   from failed changes    23
nodes added by the annotator: 5
lines added by the annotator: 11

N(c) over the 98 audited changes
  nodes: median=97 max=3892
  lines: median=295 max=5892
  files: median=5 max=57

AUROC level across the grid
  node: 0.500 to 0.975
  file: 0.512 to 0.952
  line: 0.489 to 0.974


In [18]:
ORDER = ("line", "node", "file")
LABEL = {"f1": "F1", "mcc": "MCC", "auroc": "AUROC", "map": "MAP", "mrr": "MRR"}
NOTE = {"f1": "b", "mcc": "b", "auroc": "b", "map": "c", "mrr": "c"}

for name in metrics:
    for source in ORDER:
        level = result[source][name][0]
        print(
            f"{source:5s} {name:6s} ceiling={level.max():.3f} "
            f"at auc={auc[level.argmax()]:.3f} tail_step={level[-1] - level[-2]:+.4f}"
        )


line  f1     ceiling=0.699 at auc=1.000 tail_step=+0.0000
node  f1     ceiling=0.681 at auc=1.000 tail_step=+0.0000
file  f1     ceiling=0.806 at auc=1.000 tail_step=+0.0000
line  mcc    ceiling=0.723 at auc=1.000 tail_step=+0.0000
node  mcc    ceiling=0.701 at auc=1.000 tail_step=+0.0000
file  mcc    ceiling=0.759 at auc=1.000 tail_step=+0.0000
line  auroc  ceiling=0.974 at auc=1.000 tail_step=+0.0000
node  auroc  ceiling=0.975 at auc=1.000 tail_step=+0.0000
file  auroc  ceiling=0.952 at auc=1.000 tail_step=+0.0000
line  map    ceiling=0.739 at auc=1.000 tail_step=+0.0000
node  map    ceiling=0.795 at auc=1.000 tail_step=+0.0000
file  map    ceiling=0.921 at auc=1.000 tail_step=+0.0000
line  mrr    ceiling=0.802 at auc=1.000 tail_step=+0.0000
node  mrr    ceiling=0.836 at auc=1.000 tail_step=+0.0000
file  mrr    ceiling=0.937 at auc=1.000 tail_step=+0.0000


In [22]:
# Audit-size sweep of the resolution, decomposed into two noise sources:
#
#   rate noise  -- spread from redrawing the error rates per draw, since the
#                  audit knows them only up to posterior width
#   flip noise  -- spread from the per-node Bernoulli flips at given rates
#
# Every factor f scales both Beta counters, which keeps the rates in place and
# shrinks the posterior spread by 1/sqrt(f). kappa(f) is the measured width at
# factor f; kappa(1) is the released audit. Under the two-source reading
#
#   kappa(f)^2 = flip^2 + rate^2 / f
#
# so the pairwise difference sqrt(kappa(1)^2 - kappa(f)^2) equals
# rate * sqrt(1 - 1/f): the part of the width an f-fold audit would remove.
# Both components are read off the largest measured factor, no limit involved.
# theta_star is reused unchanged, node level only, same seed everywhere so the
# detector scores are identical and the comparison is paired.
#
# Runtime: one full node-level sweep per factor plus two half-cohort runs.


def scaled(counters, factor):
    return {j: (a * factor, b * factor) for j, (a, b) in counters.items()}


FACTORS = [1, 2, 4, 8, 16, 64, 256]
TOP = FACTORS[-1]

sweep = {
    f: theta_star(
        C_node, grid, scaled(counts_node, f), scaled(gamma_node, f),
        M=M, rng=np.random.default_rng(1),
    )
    for f in FACTORS
}

band = (auc >= 0.6) & (auc <= 0.9)


def components(width_1, width_top):
    # Solve the two-source model with the measured endpoints f = 1 and f = TOP:
    # rate^2 * (1 - 1/TOP) = kappa(1)^2 - kappa(TOP)^2, then flip from either.
    rate2 = max(width_1**2 - width_top**2, 0.0) / (1.0 - 1.0 / TOP)
    flip2 = max(width_top**2 - rate2 / TOP, 0.0)
    return np.sqrt(rate2), np.sqrt(flip2)


print("two-source split, node level")
split = {}
for name in metrics:
    k1, ktop = sweep[1][name][2], sweep[TOP][name][2]
    rate, flip = components(k1, ktop)
    b1 = sweep[1][name][1][band].max()
    btop = sweep[TOP][name][1][band].max()
    b_rate, b_flip = components(b1, btop)
    split[name] = (rate, flip)
    print(
        f"{name:6s}: kappa*(1)={k1:.3f} rate={rate:.3f} flip={flip:.3f} "
        f"rate share={rate**2 / k1**2:.2f} | band kappa(1)={b1:.3f} "
        f"rate share={b_rate**2 / b1**2:.2f}"
    )


two-source split, node level
f1    : kappa*(1)=0.116 rate=0.108 flip=0.044 rate share=0.86 | band kappa(1)=0.034 rate share=0.75
mcc   : kappa*(1)=0.099 rate=0.091 flip=0.039 rate share=0.85 | band kappa(1)=0.031 rate share=0.63
auroc : kappa*(1)=0.034 rate=0.018 flip=0.028 rate share=0.30 | band kappa(1)=0.030 rate share=0.22
map   : kappa*(1)=0.093 rate=0.069 flip=0.062 rate share=0.55 | band kappa(1)=0.063 rate share=0.20
mrr   : kappa*(1)=0.091 rate=0.058 flip=0.070 rate share=0.41 | band kappa(1)=0.076 rate share=0.19


In [20]:
# Replacement for the plot section of the sweep cell. Descriptive only: the
# measured points, connected per metric. No model curve; the two-source model
# stays in the text and in the printed split. Reading: each point is the width
# an f-fold audit would remove, so a flat tail means the audit lever is
# exhausted and the remaining width is flip noise.
palette = pio.templates["thesis"].layout.colorway

fig = go.Figure()
for k, name in enumerate(metrics):
    k1 = sweep[1][name][2]
    removed = [
        np.sqrt(max(k1**2 - sweep[f][name][2] ** 2, 0.0)) for f in FACTORS[1:]
    ]
    fig.add_trace(
        go.Scatter(
            x=FACTORS[1:], y=removed, mode="lines+markers", name=name.upper(),
            line=dict(color=palette[k % len(palette)], width=1.5),
            marker=dict(size=7),
        )
    )
fig.update_xaxes(type="log", title_text="audit scale factor f")
fig.update_yaxes(title_text="width removed by an f-fold audit", rangemode="tozero")
fig.update_layout(
    template="thesis", width=800, height=420,
    title="Resolution width removed against audit size, node level",
)
fig.show(config={"toImageButtonOptions": {"format": "svg"}})

In [23]:
# Cohort dependence, the counterpart check. At factor TOP the rate noise is as
# good as gone, so the remaining width is flip noise, which is independent per
# node and should grow by about sqrt(2) on a half cohort for the pooled
# metrics. The full width at f = 1 should move far less when the rate share
# dominates, since the rate draw is shared across the corpus and does not care
# about cohort size.
half_rng = np.random.default_rng(7)
half_idx = half_rng.choice(len(C_node), size=len(C_node) // 2, replace=False)
C_half = [C_node[i] for i in sorted(half_idx)]

half = {
    f: theta_star(
        C_half, grid, scaled(counts_node, f), scaled(gamma_node, f),
        M=M, rng=np.random.default_rng(1),
    )
    for f in (1, TOP)
}

print("\nhalf cohort against full cohort, node level, kappa*")
for name in metrics:
    print(
        f"{name:6s}: f=1 {sweep[1][name][2]:.3f} -> half {half[1][name][2]:.3f} | "
        f"f={TOP} {sweep[TOP][name][2]:.3f} -> half {half[TOP][name][2]:.3f}"
    )


half cohort against full cohort, node level, kappa*
f1    : f=1 0.116 -> half 0.124 | f=256 0.044 -> half 0.057
mcc   : f=1 0.099 -> half 0.108 | f=256 0.039 -> half 0.051
auroc : f=1 0.034 -> half 0.040 | f=256 0.028 -> half 0.038
map   : f=1 0.093 -> half 0.109 | f=256 0.062 -> half 0.088
mrr   : f=1 0.091 -> half 0.117 | f=256 0.070 -> half 0.100


In [24]:
# Check: the ceiling must not move across the sweep. The counter scaling keeps
# the rate means in place, so the level curve (mean metric over the draws) may
# differ between factors only by the shrinking prior offset and Monte Carlo
# jitter. Reported per metric: the largest absolute gap between the f = 1 and
# f = TOP level curves over the whole grid, and the same at the grid point
# closest to AUC 0.8. Anything beyond ~0.005 would need a closer look.
print("ceiling drift across the sweep, node level")
probe = int(np.argmin(np.abs(auc - 0.8)))
for name in metrics:
    lvl_1 = sweep[1][name][0]
    lvl_top = sweep[TOP][name][0]
    gap = np.nanmax(np.abs(lvl_1 - lvl_top))
    print(
        f"{name:6s}: max |level(1) - level({TOP})| = {gap:.4f} | "
        f"at AUC 0.8: {lvl_1[probe]:.4f} vs {lvl_top[probe]:.4f}"
    )

ceiling drift across the sweep, node level
f1    : max |level(1) - level(256)| = 0.0064 | at AUC 0.8: 0.0818 vs 0.0819
mcc   : max |level(1) - level(256)| = 0.0079 | at AUC 0.8: 0.1173 vs 0.1184
auroc : max |level(1) - level(256)| = 0.0082 | at AUC 0.8: 0.7842 vs 0.7889
map   : max |level(1) - level(256)| = 0.0115 | at AUC 0.8: 0.2716 vs 0.2740
mrr   : max |level(1) - level(256)| = 0.0057 | at AUC 0.8: 0.3385 vs 0.3405


In [25]:
# Dropout of the per-change metrics: a change whose perturbed label set comes
# out empty carries no MAP/MRR value in that draw. theta_star already counts
# the survivors per draw and returns them as "evaluable": (mean, min, max of
# changes remaining per draw, total changes). Printed across the sweep factors,
# read on the existing sweep dictionary, no new runs.
#
# Reading: total - mean is the average dropout per draw. The min-max span is
# how much the effective cohort wobbles between draws; the part of that span
# that shrinks from f = 1 to f = 256 hung on the rate draw, the part that
# remains is per-node flip. Both shrink the rate share of MAP and MRR relative
# to the pooled metrics, which never drop a change.
print("evaluable changes per draw across the sweep (per-change metrics)")
for f in FACTORS:
    mean, low, high, total = sweep[f]["evaluable"]
    print(
        f"f={f:>4d}: mean {mean:6.1f} of {total}  "
        f"range {low:.0f}-{high:.0f}  span {high - low:.0f}"
    )

evaluable changes per draw across the sweep (per-change metrics)
f=   1: mean  263.8 of 360  range 227-301  span 74
f=   2: mean  263.7 of 360  range 232-292  span 60
f=   4: mean  263.5 of 360  range 237-292  span 55
f=   8: mean  263.1 of 360  range 241-287  span 46
f=  16: mean  262.8 of 360  range 238-289  span 51
f=  64: mean  263.2 of 360  range 230-288  span 58
f= 256: mean  263.1 of 360  range 230-288  span 58
